# 01 · Trace paired data and input-only preprocessing


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Establish the population before rendering

AMASS stores motions and body-model parameters, which supply paired projected
references after rendering. GAVD stores real videos with sequence-level gait
and camera annotations; these labels support a separate observational analysis
and cannot serve as ground-truth joint coordinates.

The AMASS cohort joins eligible motion paths to audited person identities and
the original person splits. A subject folder is not necessarily a distinct
person: activity folders can be aliases. A motion is eligible for a window
only if its duration is at least $(T-1)/h$, where $T$ is the sample count and
$h$ is the requested sampling rate. At 128 frames and 25 Hz this is 5.08 s.
The saved plan records every candidate and exclusion. Filenames identifying
walking are a reproducible candidate screen; they are not a clinical diagnosis
or a substitute for inspecting the actual movement.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
config = study.artifact('config.json')
minimum_span = (config['data']['samples'] - 1) / config['data']['hz']
print('Required window span (seconds):', minimum_span)
if study.fixture:
    print('Generated software fixture: no source inventory or clinical evidence.')
elif config['data'].get('source_selection', 'legacy_roster') == 'full_manifest':
    manifest_dir = Path(config['preparation']['manifest_dir'])
    inventory = pd.read_csv(manifest_dir / 'amass_raw_inventory_eligible.csv')
    registry = pd.read_csv(manifest_dir / 'amass_subject_registry.csv', keep_default_na=False)
    splits = pd.read_csv(manifest_dir / 'amass_subject_splits.csv', keep_default_na=False)
    person_splits = splits[['identity', 'split']].drop_duplicates()
    assert person_splits.groupby('identity')['split'].nunique().max() == 1
    duration = (inventory['num_frames'] - 1) / inventory['mocap_framerate']
    display(inventory.assign(long_enough=duration >= minimum_span)
            .groupby('source_dataset').agg(motions=('relative_path', 'size'),
                                          duration_candidates=('long_enough', 'sum')))
    display(person_splits.groupby('split').size().rename('audited people'))
    cohort_path = Path(config['cohort']['plan_path'])
    cohort = json.loads(cohort_path.read_text())
    print('Frozen cohort selection:', cohort_path)
    print('Read the cohort summary and exclusion files before interpreting candidate counts.')
    print(json.dumps(cohort.get('summary', {}), indent=2))
else:
    print('Explicit legacy roster:', config['source_bundle'])


## Keep real-video groups intact

GAVD sequences from the same recording can overlap or show the same person
from different directions. Partition complete video groups, and merge groups
when a reviewed identity map establishes that they share a person. Unknown
identity is recorded rather than treated as proof that videos show different
people. The fields `dataset_annotation` and `gait_pattern_annotation` describe
different annotation levels; use the latter for the declared gait-pattern task.
Camera labels such as `left side` describe view direction, not affected side.

The optional GAVD plan below inventories the complete manifest and local video
availability. Its planned test groups stay locked. Run `gavd-plan`, then
`launch --gavd-only` from the HAIC guide to extract train/development tracks
through the same GPU queue used for AMASS. Run these stages sequentially; a
second stage request while a coordinator is active does not enqueue that stage.


In [ ]:
gavd_plan_path = study.work / 'gavd/plan.json'
if gavd_plan_path.exists():
    real_plan = json.loads(gavd_plan_path.read_text())
    print('GAVD plan:', gavd_plan_path)
    print(json.dumps(real_plan.get('summary', {}), indent=2))
elif not study.fixture:
    import shlex
    print(shlex.join(['bash', str(study.root / 'slurm/gait-fidelity/run.sh'),
                     'gavd-plan', str(study.work)]))
    print('This CPU inventory does not claim that real-video reference coordinates exist.')
else:
    print('GAVD data are not synthesized by the teaching fixture.')


## Prepare the shared data before inspecting its mathematics

An **estimated/reference pair** contains two trajectories of the same joints at
the same instants: the pose estimator supplies the input, while the projected
body-model joints supply the training target. A **movement pair** contains two
such examples with different movement states. We use the first pairing for pose
restoration and the second to ask whether restoration preserves a change in
movement. The cells below keep these two meanings separate.

Preparation remains the production operation because it includes licensed body
assets, rendering, extraction, and immutable provenance. The fixture runs here
on the CPU. For source data, run the printed Slurm command on HAIC and wait for
preparation to finish before executing the next cell. The remaining examples
read the prepared bundle and work on small copies; they do not change it.


In [ ]:
if study.fixture:
    study.command('prepare')
else:
    import shlex
    command = ['bash', str(study.root / 'slurm/gait-fidelity/run.sh'),
               'launch', str(study.work), '--prepare-only']
    print('Run on HAIC, then continue after preparation completes:')
    print(shlex.join(command))


In [ ]:
study.command('validate')
import numpy as np
import pandas as pd
from IPython.display import display
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
from gavd6_sjepa.research_directions.synthetic_training_v2.contracts import JOINTS

bundle = load_dataset(study.bundle_path())
records = pd.DataFrame(bundle.records)
config = study.artifact('config.json')
display(records.groupby(['split', 'extractor']).size().rename('track records').reset_index())
display(records[['split', 'canonical_person_id']].drop_duplicates()
        .groupby('split').size().rename('independent people'))
print('Evidence:', bundle.evidence_status)
print('All demonstrations below select training rows by metadata, before examining error.')


## Follow one aligned example through its arrays

The batch dimension selects examples, the time dimension selects frames, and
the joint dimension follows the fixed body-12 order printed below. Input
coordinates are measured in pixels. A native confidence score comes from the
pose estimator; it is not assumed to be a calibrated probability. The boolean
`observed` array states whether a joint position was supplied. Missing input
positions remain missing rather than being interpolated.

References are stored separately. `valid` records reference availability and
`visible` records the synthetic visibility proxy. A joint may have a valid
reference while being hidden in the image. `eval_scale` is a reference-derived
box diagonal used for coordinate-error evaluation; it is never an input to the
restoration model. Person IDs, movement labels, camera IDs, and reference masks
are likewise absent from inference inputs.


In [ ]:
selected = records.index[
    records['split'].eq('train') & records['movement_state'].eq('baseline')
    & records['physical_state'].eq('original') & records['naming'].eq('correct')
    & records['observation'].eq('clear')
]
assert len(selected), 'Preparation must retain a clear, correctly named training baseline.'
row_id = int(selected[0])
raw = {key: values[[row_id]].copy() for key, values in bundle.inputs.items()}
reference = {key: values[[row_id]].copy() for key, values in bundle.targets.items()}
assert set(raw) == {'xy', 'confidence', 'observed', 'timestamps'}
assert set(reference) == {'xy', 'valid', 'visible', 'eval_scale'}
B, T, J, D = raw['xy'].shape
assert (B, J, D) == (1, 12, 2)
assert raw['observed'].shape == reference['valid'].shape == (B, T, J)
assert reference['xy'].shape == raw['xy'].shape
assert np.all(~reference['visible'] | reference['valid'])
assert np.isfinite(raw['xy'][raw['observed']]).all()
assert np.allclose(np.diff(raw['timestamps'], axis=1), 1 / config['data']['hz'])

display(pd.DataFrame({'joint index': range(J), 'joint name': JOINTS}))
display(pd.DataFrame([
    {'role': role, 'field': key, 'shape': str(value.shape), 'dtype': str(value.dtype)}
    for role, values in [('input', raw), ('reference', reference)]
    for key, value in values.items()
]))
print('Selected metadata:', records.loc[row_id, ['canonical_person_id', 'source_family_id',
      'movement_state', 'camera_id', 'observation', 'extractor']].to_dict())
print('Window span:', float(raw['timestamps'][0, -1] - raw['timestamps'][0, 0]), 'seconds')


## See how AMASS supplies a reference in the camera image

For source data, AMASS animates a body model in three dimensions. Rendering and
reference projection use the same fixed camera. Write its camera-to-world
rotation as $R$ and position as $c$. For row-vector world points $X$, camera
coordinates are $X_c=(X-c)R$. This renderer looks along the camera's negative
$z$ axis, so positive depth is $d=-X_{c,z}$. With vertical field of view $v$,
image height $H$, and width $W$, the pixel projection is

$$f=\frac{H}{2\tan(v/2)},\qquad
u=f\frac{X_{c,x}}{d}+\frac{W-1}{2},\qquad
w=-f\frac{X_{c,y}}{d}+\frac{H-1}{2}.$$

The minus sign in the second coordinate makes image rows increase downward.
The half-pixel convention matches the renderer's array coordinates. The tiny
three-point example below verifies this projection against production code;
it does not replace an AMASS recording or demonstrate rendering realism.
Source preparation additionally rejects near-plane failures and clipped
physical variants, and freezes each camera across the paired movement states.


In [ ]:
from gavd6_sjepa.research_directions.synthetic_training.rendering import project_points

world = np.array([[0., 0., -3.], [.2, .4, -3.], [-.2, -.4, -3.]])
camera_pose = np.eye(4)  # illustrative camera at the origin, looking along -z
width, height, vertical_fov = 640, 480, np.deg2rad(50.)
camera = (world - camera_pose[:3, 3]) @ camera_pose[:3, :3]
depth = -camera[:, 2]
assert np.all(depth > .05)
focal = height / (2 * np.tan(vertical_fov / 2))
projected = np.column_stack([
    focal * camera[:, 0] / depth + (width - 1) / 2,
    -focal * camera[:, 1] / depth + (height - 1) / 2,
]).astype(np.float32)
production_xy, production_depth = project_points(world, camera_pose, width, height, vertical_fov)
np.testing.assert_allclose(projected, production_xy, rtol=0, atol=1e-5)
np.testing.assert_allclose(depth, production_depth)
display(pd.DataFrame(projected, columns=['horizontal pixel u', 'vertical pixel w']))


The reference is the projected body model's joint center. Its anatomical
definition is an approximation to the estimator's corresponding landmark,
which can introduce systematic offsets. Alignment in time and camera does not
make this a direct measurement of anatomical truth. Real-video evaluation
requires independent references and its own landmark checks.

## Inspect the second kind of pair: a controlled movement change

A `pair_id` fixes the source window, person, camera, observation condition,
joint-naming condition, and extractor while allowing the movement state to
change. The source preparation composes a local right-knee rotation during
reference-supported swing frames; its requested edit in degrees is an input
to the simulator, **not** the resulting image-plane excursion difference.
The latter must be measured from the projected references in notebook 05.
The analytic software fixture only mimics these relationships for code tests.


In [ ]:
pair_rows = records.index[records['pair_id'].eq(records.loc[row_id, 'pair_id'])].to_numpy()
pair = records.loc[pair_rows]
for field in ['canonical_person_id', 'source_family_id', 'split', 'physical_state',
              'camera_id', 'observation', 'naming', 'extractor']:
    assert pair[field].nunique() == 1, f'Uncontrolled change in {field}'
for k in pair_rows:
    np.testing.assert_array_equal(bundle.inputs['timestamps'][k], raw['timestamps'][0])
display(pair[['movement_state', 'movement_magnitude', 'camera_id', 'observation', 'naming']])

# The exact duplicate is retained as a zero-change diagnostic, not another person.
unchanged_rows = pair.index[pair['movement_state'].eq('no_change')]
assert len(unchanged_rows) == 1
np.testing.assert_array_equal(bundle.targets['xy'][unchanged_rows[0]], reference['xy'][0])
print('The no-change reference is exactly equal to its baseline.')


## Normalize using only the context that the encoder may see

Let $C$ contain the observed joint positions remaining after artificial query
masking. For each window, production computes a two-coordinate origin $o$ and
one positive scale $s$:

$$o=\operatorname{median}_{x\in C}x,\qquad
s=\lVert Q_{.95}(C)-Q_{.05}(C)\rVert_2,\qquad
\widetilde{x}=(x-o)/s.$$

The quantiles act independently on horizontal and vertical coordinates; their
difference forms a diagonal whose Euclidean length is the common scale.
Using one scale preserves angles and relative geometry. Both quantities are
constant throughout the window. If fewer than two context points remain, or
their span is degenerate, production uses $o=(0,0)$ and $s=1$ and counts the
fallback. Neither hidden values nor references determine this fallback.

The next cell shows the complete computation. We hide the first four-frame
patch of both knees as a deterministic teaching example. Notebook 02 replaces
this illustrative mask with the study's stochastic sampler.


In [ ]:
patch_size = int(config['model']['patch_size'])
assert T % patch_size == 0
hidden = np.zeros((B, T // patch_size, J), dtype=bool)
hidden[:, 0, [8, 9]] = True
hidden_frames = np.repeat(hidden, patch_size, axis=1)

def normalize_from_context(inputs, hidden_frames):
    xy = np.asarray(inputs['xy'], dtype=np.float32)
    context = inputs['observed'] & ~hidden_frames
    origins, scales, fallbacks = [], [], 0
    for points, keep in zip(xy, context):
        retained = points[keep]
        if len(retained) >= 2:
            origin = np.median(retained, axis=0)
            span = np.quantile(retained, .95, axis=0) - np.quantile(retained, .05, axis=0)
            scale = float(np.linalg.norm(span))
        else:
            origin, scale = np.zeros(2), 0.
        if not np.isfinite(scale) or scale < 1e-6:
            origin, scale = np.zeros(2), 1.
            fallbacks += 1
        origins.append(origin)
        scales.append(scale)
    origins = np.asarray(origins, dtype=np.float32)
    scales = np.asarray(scales, dtype=np.float32)
    normalized = {key: values.copy() for key, values in inputs.items()}
    normalized['xy'] = (xy - origins[:, None, None]) / scales[:, None, None, None]
    normalized['confidence'] = np.where(np.isfinite(inputs['confidence']),
                                        inputs['confidence'], 0).astype(np.float32)
    return normalized, origins, scales, fallbacks

normalized, origin, scale, fallback_count = normalize_from_context(raw, hidden_frames)
from gavd6_sjepa.research_directions.gait_fidelity.training import normalize_batch
actual, actual_origin, actual_scale, actual_fallbacks = normalize_batch(raw, hidden, patch_size=patch_size)
for key in normalized:
    np.testing.assert_allclose(normalized[key], actual[key], rtol=0, atol=0, equal_nan=True)
np.testing.assert_array_equal(origin, actual_origin)
np.testing.assert_array_equal(scale, actual_scale)
assert fallback_count == actual_fallbacks
print({'origin_px': origin.tolist(), 'scale_px': scale.tolist(), 'fallbacks': fallback_count})


### Check the leakage boundary and restore pixel geometry

Changing a hidden input must leave the normalization and visible context
unchanged. We test that property by moving the hidden coordinates a million
pixels, then rebuilding the context channels. The normalized hidden values
may differ in memory, but token construction replaces them with zero before
the encoder sees them. References are transformed by the **same input-derived**
$o,s$ for the training target; references never provide their own normalization.

The model predicts normalized positions. Applying
$\widehat{x}=s\widehat{\widetilde{x}}+o$ returns them to pixels before gait
measurements and the minimum limb-length check. This cell verifies the inverse
on the original observations, without claiming an improvement from a model.


In [ ]:
perturbed = {key: values.copy() for key, values in raw.items()}
perturbed['xy'][hidden_frames & raw['observed']] += 1_000_000
altered, altered_origin, altered_scale, _ = normalize_from_context(perturbed, hidden_frames)
np.testing.assert_array_equal(origin, altered_origin)
np.testing.assert_array_equal(scale, altered_scale)
context = raw['observed'] & ~hidden_frames
safe_context = np.where(context[..., None], normalized['xy'], 0.)
altered_context = np.where(context[..., None], altered['xy'], 0.)
np.testing.assert_array_equal(safe_context, altered_context)

target_normalized = (reference['xy'] - origin[:, None, None]) / scale[:, None, None, None]
restored_input_px = normalized['xy'] * scale[:, None, None, None] + origin[:, None, None]
restored_reference_px = target_normalized * scale[:, None, None, None] + origin[:, None, None]
np.testing.assert_allclose(restored_input_px[raw['observed']], raw['xy'][raw['observed']],
                           rtol=2e-6, atol=1e-4)
np.testing.assert_allclose(restored_reference_px[reference['valid']],
                           reference['xy'][reference['valid']], rtol=2e-6, atol=1e-4)

empty = {key: values.copy() for key, values in raw.items()}
empty['observed'][:] = False
empty['xy'][:] = np.nan
_, empty_origin, empty_scale, empty_count = normalize_from_context(empty, hidden_frames)
assert empty_count == B
np.testing.assert_array_equal(empty_origin, np.zeros((B, 2)))
np.testing.assert_array_equal(empty_scale, np.ones(B))
_, production_empty_origin, production_empty_scale, production_empty_count = normalize_batch(
    empty, hidden, patch_size=patch_size)
np.testing.assert_array_equal(empty_origin, production_empty_origin)
np.testing.assert_array_equal(empty_scale, production_empty_scale)
assert production_empty_count == empty_count
print('Hidden-coordinate perturbation, pixel round trip, and empty-context fallback passed.')


## Inspect positions and trajectories before fitting

The following picture uses the selected training row's middle frame and its
complete ankle trajectory. The choice is made from metadata and frame index,
without ranking prediction errors. Blue points/curves are references; orange
ones are the estimator inputs. The connecting lines describe body-12 geometry.
These two panels are data checks and contain no trained restoration prediction.
The extra previews below may include retained evaluation figures when reopening
an already completed study; their labels identify what they summarize.


In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image
from gavd6_sjepa.research_directions.gait_fidelity.masking import EDGES

frame = T // 2
seconds = raw['timestamps'][0] - raw['timestamps'][0, 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
for points, color, label in [(reference['xy'][0, frame], '#2468a0', 'Reference'),
                             (raw['xy'][0, frame], '#cb7022', 'Estimator input')]:
    for a, b in EDGES:
        axes[0].plot(points[[a, b], 0], points[[a, b], 1], color=color, alpha=.65)
    axes[0].scatter(points[:, 0], points[:, 1], s=22, color=color, label=label)
axes[0].set(xlabel='Horizontal position (pixels)', ylabel='Vertical position (pixels)',
            title=f'Middle frame: {seconds[frame]:.2f} seconds')
axes[0].invert_yaxis()
axes[0].set_aspect('equal', adjustable='datalim')
axes[0].legend(loc='best', fontsize=9)
axes[1].plot(seconds, reference['xy'][0, :, 11, 0], color='#2468a0', label='Reference')
axes[1].plot(seconds, raw['xy'][0, :, 11, 0], color='#cb7022', label='Estimator input')
axes[1].set(xlabel='Time since window start (s)', ylabel='Horizontal position (pixels)',
            title='Right ankle through time')
axes[1].legend(loc='best', fontsize=9)
fig.suptitle(('Software fixture' if study.fixture else 'Selected training example') +
             ' — aligned reference and input')
png = BytesIO()
fig.savefig(png, format='png', dpi=120, bbox_inches='tight')
display(Image(data=png.getvalue()))
plt.close(fig)

for path in dict.fromkeys(sorted(study.work.rglob('*viewer*.html')) +
                          sorted(study.work.rglob('*gallery*.html'))):
    print('Interactive prepared-data viewer:', path)
preview_images(study)


The production viewer also exposes camera, movement, visibility, and naming
conditions. Check every declared condition and retained failure before fitting.
All variants of a person remain in that person's original split; render counts
do not increase the number of independent people. Full-manifest selection
retains all relevant audited candidates under the saved duration and cohort
rules; actual admitted counts come from this run's preparation report.
Original test people remain locked while development choices are made.
Continue to **02_masking_and_controls**
to turn these arrays into query masks and model tokens.
